In [ ]:
%env AWS_PROFILE=kmnlp

In [ ]:
import sys

sys.path.append("../src")

In [ ]:
import math
import re
from datetime import datetime
from os.path import join
from typing import cast

import fsspec  # pyright: ignore[reportMissingTypeStubs]
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from pydantic import ValidationError
from tqdm import tqdm

from llm_agent.evaluation.base_eval import EvalReport
from llm_agent.evaluation.utils import SummaryStats
from llm_agent.utils.utils import model_from_file


In [ ]:
sns.reset_defaults()
sns.set_theme(style="whitegrid", palette=None)

In [ ]:
# evals_root = "s3://llm-task-evals-897abd67-68fa-4724-a9df-ad3b70ef282c/"
evals_root = "/home/ahassan/noaa-kmnlp/llm-agent/evals/outputs/"

---


In [ ]:
def get_task_eval_uris(
    evals_root: str,
    *,
    task_id: str,
    dataset_name: str = "default",
    dataset_version: str | None = None,
) -> list[str]:
    task_evals_root = join(evals_root, task_id, "dataset", dataset_name)
    if dataset_version is not None:
        task_evals_root = join(task_evals_root, "version", dataset_version)
    fs, task_evals_root = cast(
        "tuple[fsspec.AbstractFileSystem, str]",
        fsspec.url_to_fs(task_evals_root),  # pyright: ignore[reportUnknownMemberType]
    )
    glob_path = join(task_evals_root, "**/aggregated.json")
    return [fs.unstrip_protocol(path) for path in fs.glob(glob_path)]  # pyright: ignore[reportArgumentType, reportUnknownVariableType, reportUnknownMemberType]

In [ ]:
uri_regex = re.compile(
    r"^(?:(?P<protocol>[^:/]+)://)?(?P<root>[\w\d\-_/]+)/(?P<task_id>[^/]+)/dataset/(?P<dataset_id>[^/]+)/version/(?P<dataset_version>[^/]+)/(?P<exp_name>[^/]+)/(?P<timestamp>\d{4}-\d{2}-\d{2}T\d{4})(?:_(?P<tag>[^/]+))?/aggregated.json"
)


def extract_metrics(
    aggregated_metrics: dict[str, int | float | SummaryStats],
) -> dict[str, int | float | SummaryStats]:
    out: dict[str, int | float | SummaryStats] = {}
    for k, v in aggregated_metrics.items():
        try:
            stats = SummaryStats.model_validate(v)
            out[f"{k}_mean"] = stats.mean
            out[f"{k}_stderr"] = stats.std / math.sqrt(len(stats.values))
        except ValidationError:
            out[k] = v
    return out


def parse_uri(uri: str) -> dict[str, str]:
    match = uri_regex.fullmatch(uri)
    if match is None:
        raise ValueError
    parsed = match.groupdict()
    parsed.pop("protocol")
    parsed.pop("root")
    parsed["timestamp"] = datetime.strptime(parsed["timestamp"], "%Y-%m-%dT%H%M")  # noqa: DTZ007
    return parsed


def extract_record(uri: str) -> dict[str, str | int | float | SummaryStats]:
    exp_info = parse_uri(uri)

    eval_report: EvalReport = model_from_file(uri, EvalReport)
    assert exp_info["dataset_id"] == eval_report.experiment_config.dataset_id
    assert exp_info["dataset_version"] == eval_report.experiment_config.dataset_version
    assert exp_info["exp_name"] == eval_report.experiment_config.name, (
        f"Expected {exp_info['exp_name']}, got {eval_report.experiment_config.name}"
    )

    metrics = extract_metrics(eval_report.aggregated_metrics)
    return {**exp_info, **metrics}


def get_records_for_task(
    evals_root: str,
    *,
    task_id: str,
    dataset_name: str = "default",
    dataset_version: str | None = None,
) -> list[dict[str, str | int | float | SummaryStats]]:
    uris = get_task_eval_uris(
        evals_root,
        task_id=task_id,
        dataset_name=dataset_name,
        dataset_version=dataset_version,
    )
    with tqdm(uris) as bar:
        return [extract_record(uri) for uri in bar]

---


## Algorithm Agent


In [ ]:
df_alg = pd.DataFrame.from_records(
    get_records_for_task(
        evals_root,
        task_id="algorithm_agent",
        dataset_name="default",
        dataset_version="6",
    )
)
df_alg.loc[:, "pct_correct_95_ci"] = df_alg["pct_correct_stderr"] * 1.96
df_alg

# Flat Bar Chart


In [ ]:
import matplotlib.colors as mcolors

# "<model>_no_retry" is the baseline, "<model>" adds retries, "<model>_thinking" adds thinking too
model_order = ["claude_4_5_haiku", "claude_4_6_sonnet", "claude_4_6_opus"]
model_names = {
    "claude_4_5_haiku": "Haiku 4.5",
    "claude_4_6_sonnet": "Sonnet 4.6",
    "claude_4_6_opus": "Opus 4.6",
}
model_colors = {
    "claude_4_5_haiku": "#5b9bd5",  # blue
    "claude_4_6_sonnet": "#4ab3a0",  # teal
    "claude_4_6_opus": "#f0916a",  # orange
}
# variant -> (order within model, label suffix, shade toward the model's base color)
variants = {
    "no_retry": (0, "{model}", 0.35),
    "": (1, "{model}\n(retry)", 0.65),
    "thinking": (2, "{model}\n(retry + thinking)", 1.0),
}


def split_exp_name(exp_name: str) -> tuple[str, str]:
    for suffix in ("_no_retry", "_thinking"):
        if exp_name.endswith(suffix):
            return exp_name[: -len(suffix)], suffix[1:]
    return exp_name, ""


def shade_color(hex_color: str, shade: float) -> tuple[float, float, float]:
    base = mcolors.to_rgb(hex_color)
    return tuple(1 * (1 - shade) + b * shade for b in base)  # pyright: ignore[reportReturnType]


order: dict[str, int] = {}
labels: dict[str, str] = {}
palette: dict[str, tuple[float, float, float]] = {}
for exp_name in df_alg["exp_name"].unique():
    model, variant = split_exp_name(exp_name)
    variant_rank, label_fmt, shade = variants[variant]
    order[exp_name] = model_order.index(model) * len(variants) + variant_rank
    labels[exp_name] = label_fmt.format(model=model_names[model])
    palette[exp_name] = shade_color(model_colors[model], shade)

df_alg = df_alg.sort_values(by="exp_name", key=lambda x: x.map(order))

fig, ax = plt.subplots(figsize=(14, 6))  # pyright: ignore[reportUnknownMemberType]

sns.barplot(
    data=df_alg,
    x="exp_name",
    y="pct_correct_mean",
    errorbar=None,
    width=0.65,
    edgecolor="white",
    linewidth=1,
    ax=ax,
)
# Color each bar individually (single container keeps bar_label working below)
for patch, exp_name in zip(ax.patches, df_alg["exp_name"], strict=True):
    patch.set_facecolor(palette[exp_name])  # pyright: ignore[reportUnknownMemberType, reportAttributeAccessIssue]

ax.errorbar(  # pyright: ignore[reportUnknownMemberType]
    x=df_alg["exp_name"],
    y=df_alg["pct_correct_mean"],
    yerr=df_alg["pct_correct_95_ci"],
    fmt="none",
    c="#333333",  # Error bar color
    capsize=5,  # Width of the horizontal caps
    lw=1.5,  # Line width
)

# Place labels above the error bar cap (not just the bar) so they never overlap it
for x_pos, (mean, ci) in enumerate(
    zip(df_alg["pct_correct_mean"], df_alg["pct_correct_95_ci"], strict=True)
):
    ax.text(  # pyright: ignore[reportUnknownMemberType]
        x_pos,
        mean + ci + 1.2,
        f"{mean:.1f} ± {ci:.1f}",
        ha="center",
        va="bottom",
        color="#222222",
        fontsize=11,
        fontweight="bold",
    )

ax.set_xticks(range(len(df_alg)))  # pyright: ignore[reportUnknownMemberType]
ax.set_xticklabels(  # pyright: ignore[reportUnknownMemberType]
    [labels[exp] for exp in df_alg["exp_name"]], fontsize=10
)
ax.set_ylabel("Correct (%)", fontsize=12)  # pyright: ignore[reportUnknownMemberType]
ax.set_xlabel("")  # pyright: ignore[reportUnknownMemberType]
ax.set_title(  # pyright: ignore[reportUnknownMemberType]
    "Algorithm Agent Accuracy by Model\nMean ± 95% CI over 3 runs per experiment",
    fontsize=13,
    fontweight="bold",
    pad=14,
)
ax.set_ylim((50, 100))
ax.grid(axis="x", visible=False)  # pyright: ignore[reportUnknownMemberType]
ax.grid(axis="y", alpha=0.3)  # pyright: ignore[reportUnknownMemberType]
sns.despine(ax=ax, left=True, bottom=True)
fig.tight_layout()
plt.show()  # pyright: ignore[reportUnknownMemberType]


# Grouped Bar Chart


In [ ]:
from matplotlib.patches import Patch

model_colors = {
    "claude_4_5_haiku": "#5b9bd5",  # blue
    "claude_4_6_sonnet": "#4ab3a0",  # teal
    "claude_4_6_opus": "#f0916a",  # orange
}

# Grouped bar chart: one group per model, variants differentiated by hatch (same color per model)
variant_order_list = ["no_retry", "", "thinking"]
variant_labels = {
    "no_retry": "Baseline",
    "": "With retries",
    "thinking": "With retries +\nthinking",
}
variant_hatches = {
    "no_retry": "",
    "": "/",
    "thinking": "x",
}

plot_df = df_alg.copy()
plot_df["model"], plot_df["variant"] = zip(*plot_df["exp_name"].map(split_exp_name), strict=True)
plot_df["model_label"] = plot_df["model"].map(model_names)
model_label_order = [model_names[m] for m in model_order]

fig2, ax2 = plt.subplots(figsize=(10, 6))  # pyright: ignore[reportUnknownMemberType]

sns.barplot(
    data=plot_df,
    x="model_label",
    y="pct_correct_mean",
    hue="variant",
    order=model_label_order,
    hue_order=variant_order_list,
    errorbar=None,
    width=0.7,
    linewidth=1.2,
    legend=False,
    ax=ax2,
)

# Recolor by model (not variant) and apply a hatch per variant; each container is one variant,
# with one bar per model in `model_label_order`
error_bar_positions: list[tuple[float, float, float]] = []
for container, variant in zip(ax2.containers, variant_order_list, strict=True):
    for patch, model in zip(container, model_order, strict=True):  # pyright: ignore[reportUnknownVariableType]
        patch.set_facecolor(model_colors[model])  # pyright: ignore[reportUnknownMemberType, reportAttributeAccessIssue]
        patch.set_hatch(variant_hatches[variant])  # pyright: ignore[reportUnknownMemberType, reportAttributeAccessIssue]
        patch.set_edgecolor((0, 0, 0, 0.5))  # pyright: ignore[reportUnknownMemberType, reportAttributeAccessIssue]

        row = plot_df[(plot_df["model"] == model) & (plot_df["variant"] == variant)].iloc[0]
        x_pos = patch.get_x() + patch.get_width() / 2  # pyright: ignore[reportUnknownVariableType, reportUnknownMemberType, reportAttributeAccessIssue]
        error_bar_positions.append((x_pos, row["pct_correct_mean"], row["pct_correct_95_ci"]))  # pyright: ignore[reportUnknownArgumentType]

xs, means, cis = zip(*error_bar_positions, strict=True)
ax2.errorbar(  # pyright: ignore[reportUnknownMemberType]
    x=xs,
    y=means,
    yerr=cis,
    fmt="none",
    c="#333333",
    capsize=4,
    lw=2,
)

for x_pos, mean, ci in error_bar_positions:
    ax2.text(  # pyright: ignore[reportUnknownMemberType]
        x_pos,
        mean + ci + 1.2,
        f"{mean:.1f}",
        ha="center",
        va="bottom",
        color="#222222",
        fontsize=9,
        fontweight="bold",
    )

legend_handles = [
    Patch(
        facecolor="white",
        edgecolor="black",
        hatch=variant_hatches[variant] * 2,
        label=variant_labels[variant],
    )
    for variant in variant_order_list
]
ax2.legend(  # pyright: ignore[reportUnknownMemberType]
    handles=legend_handles,
    title="Variant",
    loc="upper left",
    bbox_to_anchor=(1.01, 1),
    frameon=False,
    fontsize=12,
)

ax2.set_ylabel("Correct (%)", fontsize=12)  # pyright: ignore[reportUnknownMemberType]
ax2.set_xlabel("")  # pyright: ignore[reportUnknownMemberType]
ax2.set_title(  # pyright: ignore[reportUnknownMemberType]
    "Algorithm Agent Accuracy by Model\nMean ± 95% CI over 3 runs per experiment",
    fontsize=13,
    fontweight="bold",
    pad=14,
)
ax2.set_ylim((50, 100))
ax2.grid(axis="x", visible=False)  # pyright: ignore[reportUnknownMemberType]
ax2.grid(axis="y", alpha=0.3)  # pyright: ignore[reportUnknownMemberType]
sns.despine(ax=ax2, left=True, bottom=True)
fig2.tight_layout()
plt.show()  # pyright: ignore[reportUnknownMemberType]
